# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their `@id` as required by the dataset's schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their IDs, and fields.

Each RecordSet can be referenced by `@id`. Below, we enumerate all available record sets and the fields (columns) within them. By convention, the dataset schema uses the `@id` fields starting with `cr:RecordSet` for record sets and `cr:Field` for fields.

In [ ]:
# List all available RecordSet IDs and fields in the dataset
record_set_ids = []
field_ids_per_record_set = {}
for recordset in metadata.record_sets:
    print(f"RecordSet name: {recordset.name}  |  @id: {recordset.id}")
    record_set_ids.append(recordset.id)
    # list fields for this record set
    field_ids = []
    print("  Fields in this RecordSet:")
    for field in recordset.fields:
        print(f"    - {field.name}  |  @id: {field.id}  | dataType: {field.data_type}")
        field_ids.append(field.id)
    field_ids_per_record_set[recordset.id] = field_ids
    print()
# Optionally, display all record set ids for future use
print(f"All RecordSet IDs: {record_set_ids}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. We use the record set and field `@id`s.

`mlcroissant.Dataset.records(record_set=...)` can be used to iterate records in a record set. Let's load the data for all record sets present.

In [ ]:
# Extract all record sets (by @id) into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows.")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  -- No records found for this RecordSet.")
print(f"All loaded DataFrame keys: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by attributes, referencing columns by their `@id`.

Below, we demonstrate EDA on the **first RecordSet** with available data.

In [ ]:
# Choose the first non-empty DataFrame (first record set with records)
analyzed_record_set_id = None
for rec_id in record_set_ids:
    if rec_id in dataframes and not dataframes[rec_id].empty:
        analyzed_record_set_id = rec_id
        break

if analyzed_record_set_id is not None:
    df = dataframes[analyzed_record_set_id].copy()
    print(f"Analyzing RecordSet: {analyzed_record_set_id}")
    # Print all columns (these are usually field @ids)
    print(f"Columns: {df.columns.tolist()}")
    # Choose first numeric field (by examining metadata)
    numeric_field_id = None
    group_field_id = None
    for field in metadata["record_sets_dict"][analyzed_record_set_id].fields:
        if field.data_type in ('Float', 'Integer', 'Number') and field.id in df.columns:
            numeric_field_id = field.id
            break
    for field in metadata["record_sets_dict"][analyzed_record_set_id].fields:
        # Select the first Categorical/Text field for grouping
        if field.data_type in ('Text',) and field.id in df.columns:
            group_field_id = field.id
            break
    if numeric_field_id is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        # Remove NaNs, set a threshold for filtering
        threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].dtype.kind in 'fi' else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold} (upper quartile): {len(filtered_df)} records out of {len(df)} total.")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Optionally group by another field
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped.head())
else:
    print("No non-empty record set for EDA found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

Below we plot the distribution of the selected numeric field after basic filtering (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if analyzed_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # Boxplot if grouping field available
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=30, ha="right")
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to explore a structured FAIR dataset defined by a Croissant schema. Record sets, fields, and all operations referenced entities by their `@id`, ensuring schema consistency. You can use this notebook structure to perform filtering, normalization, grouping, and visualization on any Croissant-compliant dataset.